In [7]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# OpenAI Chat Completions API 기본
이 튜토리얼은 OpenAI의 Chat Completions API를 활용하여 챗봇이나 AI 기능을 개발하는 방법을
단계별로 설명합니다. 특히 OpenAI의 최신 언어 모델 중 하나인 GPT-4o-mini/gpt-4.1-nano를 사
용하여 예제를 진행할 것입니다. 각 섹션에는 개념 설명과 함께 실행 가능한 파이썬 코드 예제가
포함되어 있습니다.

### 주요 학습 내용:
1. OpenAI API 소개 및 환경 설정: OpenAI API 개요, API 키 발급 및 보안 설정, 파이썬 클라이언트 설치 및 인스턴스 생성 방법
2. 기본적인 Chat Completions API 사용법: 간단한 대화형 텍스트 생성 요청과 응답 처리, 프롬프트 엔지니어링 기초
3. 스트리밍 응답: 대화 응답을 스트리밍 방식으로 받아 실시간 처리하는 방법
4. 시스템 메시지 활용: 시스템 역할 메시지를 사용하여 AI의 응답 스타일이나 행동을 조정하는 방법
5. 고급 활용법: 토큰 최적화와 비용 절감 전략, OpenAI API 에러 처리 및 예외Handling
6. 실전 프로젝트 예제: 간단한 챗봇 구현 및 외부 데이터/API와 연동하여 데이터 분석 기능을 결합한 사례

## 1. OpenAI API 소개 및 환경 설정
먼저 OpenAI API와 Chat Completions에 대해 간략히 알아보고, API를 사용하기 위한 환경을 설정
해보겠습니다.

### OpenAI API 개요
OpenAI API는 GPT 계열의 대규모 언어 모델을 인터넷을 통해 사용할 수 있도록 제공하는 서비스
입니다. Chat Completions API는 챗봇과 유사한 대화형 상호작용을 할 수 있는 엔드포인트로, 역할
(role)이 부여된 메시지 목록을 입력하면 모델이 다음 대화 내용을 생성합니다. GPT-4o는 텍스트와 이미지 입력을 모두 처리하며 최대 128k 토큰의 긴 문맥을 다룰 수 있습니다. GPT-4o와 경량화 모델인 GPT-4o-mini 등이 제공되며, 요구 사항에 따라 적절한 모델을 선택할 수 있습니다
(GPT-4o-mini는 비용 효율이 높음)

### API 키 발급 및 보안 설정
OpenAI API를 사용하려면 먼저 OpenAI 계정에서 API 키를 발급받아야 합니다. OpenAI 웹사이트
의 API Keys 페이지에서 새로운 비밀 키를 생성할 수 있습니다. 발급받은 API 키는 비밀로 관리해
야 하며, 소스 코드나 공개 저장소에 노출되지 않도록 주의해야 합니다. 가장 좋은 방법은 API 키
를 코드에 하드코딩하지 않고, 환경 변수나 별도의 설정 파일에 저장하는 것입니다. 이 튜토리얼
에서는 .env 파일에 키를 저장하고 파이썬에서 이를 불러오는 방식을 사용합니다. 이를 위해
Python용 패키지 **python-dotenv**를 활용하겠습니다.
- .env 파일에 키 저장: 프로젝트 디렉터리에 .env 파일을 만들고 아래와 같이 API 키를 저장합니
다 (따옴표 없이).
 ```
 OPENAI_API_KEY=발급받은-API키-값
 ```
- python-dotenv 사용: 파이썬 코드에서 python-dotenv를 이용해 .env 파일의 환경 변수를 불러
올 수 있습니다.


In [9]:
import openai
openai.__version__
# 설정 -> 개인정보 및 보안 -> 앱 및 브라우저컨트롤 -> 스마트앱컨트롤 끄기

'3.11.0'

In [12]:
from dotenv import load_dotenv
load_dotenv(
    # dotenv_path='e:/.env'
    ) # .env파일의 key와 값을 시스템 환경변수로 셋팅
import os
os.getenv('OPENAI_API_KEY')[:3]

'sk-'

In [13]:
from openai import OpenAI
client = OpenAI(
            #api_key=os.getenv('OPENAI_API_KEY')
)

In [33]:
# 설정 -> 개인정보 및 보안 -> 앱 및 브라우저컨트롤 -> 스마트앱컨트롤 끄기
response = client.responses.create(
    model="gpt-4o-mini", 
    input="Tell me a funny joke"
)
print(response.output_text)

Why don't scientists trust atoms? 

Because they make up everything!


In [34]:
response.output

[ResponseOutputMessage(id='msg_0580844dce28647c006aa7592bbabc87d08505b3d419ab760d', content=[ResponseOutputText(annotations=[], text="Why don't scientists trust atoms? \n\nBecause they make up everything!", type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]

In [18]:
print(response.output[0].content[0].text)

Why did the scarecrow win an award?

Because he was outstanding in his field!


In [20]:
response.usage

ResponseUsage(input_tokens=12, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=18, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=30)

In [21]:
response = client.responses.create(
    model="gpt-4o-mini", 
    input="웃긴 농담하나 해줘"
)
print(response.output_text)

물고기가 학교에 가면 어떤 과목을 배우는지 알아? 

"물리!" 🎣😄


In [22]:
print(response.output[0].content[0].text)

물고기가 학교에 가면 어떤 과목을 배우는지 알아? 

"물리!" 🎣😄


In [23]:
response.usage

ResponseUsage(input_tokens=15, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=26, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=41)

In [36]:
# 추론 모델
response = client.responses.create(
    model='gpt-5-nano',
    input='웃긴 농담 하나 해줘'
)
print(response.output_text)

자전거는 왜 항상 혼자 있을 수 없을까요? 항상 두 바퀴가 있어야 하니까요. 
원하시면 분위기나 스타일을 바꾼 또 다른 농담도 drôle 드릴게요!


In [44]:
response.output[0] # 추론 과정의 데이터
response.output[1] # message
response.output[1].content[0].text

'자전거는 왜 항상 혼자 있을 수 없을까요? 항상 두 바퀴가 있어야 하니까요. \n원하시면 분위기나 스타일을 바꾼 또 다른 농담도 drôle 드릴게요!'

In [24]:
class Person:
    def __init__(self):
        pass
    def output(self):
        return '결과'